## End-to-End Model Using 3D Convolutions

Utilizing a 3D Convolutional Neural Network (3D CNN) directly on the video frames. This approach eliminates the need for a separate feature  
extraction step using a pre-trained 2D CNN and an LSTM model. Instead, we will apply 3D convolutions directly on the video sequences.     
Employing three-dimensional convolutions to capture spatiotemporal patterns directly from the video data.  

In [3]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.layers import (
    Conv3D, BatchNormalization, MaxPool3D, Dense, 
    Dropout, GlobalAveragePooling3D, Input
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [4]:
class Video3DClassifier:
    def __init__(self, max_frames=10, frame_size=(224, 224)):
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.model = None

    def extract_frames(self, video_path, max_frames=10):
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error opening video file: {video_path}")
            return
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        sample_interval = total_frames // max_frames
        
        frames = []
        frame_count = 0
        sampled_count = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % sample_interval == 0:
                frame = cv2.resize(frame, (224, 224))  # 224x224 based on the pretrained model
                frames.append(frame)
                sampled_count += 1
            
            frame_count += 1
        
        cap.release()
    
        # Pad with black frames if less than max_frames
        while len(frames) < max_frames:
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
        
        return np.array(frames[:max_frames])
    
    
    def prepare_data(self, video_dir, metadata_path, max_frames):
        """Prepare dataset from videos and metadata"""
        try:
            # Read metadata with explicit data types
            dtype_dict = {
                'VideoID': str,
                'HowFunny': str  # Keep as string initially
            }
            
            metadata = pd.read_excel(
                metadata_path,
                dtype=dtype_dict
            )
            
            # Remove entries with the class "Extremely funny"
            metadata = metadata[metadata['HowFunny'] != "Extremely funny"]
            
            # Remove missing values from the HowFunny column
            metadata = metadata[metadata['HowFunny'].notna() & (metadata['HowFunny'].str.strip() != "")]
            
            # Clean the 'HowFunny' column to ensure it contains valid strings
            metadata['HowFunny'] = metadata['HowFunny'].astype(str).str.strip()
            
            # Create label mapping for string labels to integers
            unique_labels = sorted(metadata['HowFunny'].unique())
            label_to_int = {label: idx for idx, label in enumerate(unique_labels)}
            
            # Print label mapping for reference
            print("\nLabel Mapping:")
            for label, idx in label_to_int.items():
                print(f"{label} -> {idx}")
            
            videos_list = []
            labels = []
            
            print("\nProcessing videos...")
            total_videos = len(metadata)
            valid_videos = 0
            
            for idx, row in metadata.iterrows():
                try:
                    video_path = os.path.join(video_dir, str(row['VideoID']).strip())
                    if os.path.exists(video_path):
                        frames = self.extract_frames(video_path, max_frames=max_frames)
                        if frames is not None and len(frames) == self.max_frames:
                            videos_list.append(frames)
                            # Convert string label to integer using mapping
                            label_int = label_to_int[row['HowFunny']]
                            labels.append(label_int)
                            valid_videos += 1
                            
                            if valid_videos % 10 == 0:
                                print(f"Processed {valid_videos} valid videos out of {idx + 1} checked")
                    else:
                        print(f"Video not found: {video_path}")
                        
                except Exception as e:
                    print(f"Error processing video {row['VideoID']}: {str(e)}")
                    continue
            
            if not videos_list:
                raise ValueError("No valid videos processed")
                
            # Convert to numpy arrays with explicit dtypes
            X = np.array(videos_list, dtype=np.float32)
            y = np.array(labels, dtype=np.int32)
            
            # Save label mapping for later use
            self.label_mapping = {
                'int_to_label': unique_labels,
                'label_to_int': label_to_int
            }
            
            print(f"\nFinal dataset:")
            print(f"Videos shape: {X.shape}")
            print(f"Labels shape: {y.shape}")
            print("\nLabel distribution:")
            for i, count in zip(*np.unique(y, return_counts=True)):
                original_label = unique_labels[i]
                print(f"Label '{original_label}' (class {i}): {count} samples")
            
            return X, y
            
        except Exception as e:
            print(f"Error preparing dataset: {str(e)}")
            raise

    def validate_metadata(self, metadata_path):
        """Validate metadata file structure"""
        try:
            # Read the first few rows to check structure
            df = pd.read_excel(metadata_path, nrows=5)
            
            # Check required columns
            required_columns = {'VideoID', 'HowFunny'}
            missing_columns = required_columns - set(df.columns)
            
            if missing_columns:
                raise ValueError(f"Missing required columns: {missing_columns}")
                
            # Print metadata info
            print("\nMetadata Structure:")
            print(df.dtypes)
            print("\nSample data:")
            print(df.head())
            
            return True
            
        except Exception as e:
            print(f"Error validating metadata: {str(e)}")
            return False
    
    def build_model(self, num_classes):
        
        """Building 3D CNN model"""
        inputs = Input(shape=(self.max_frames, *self.frame_size, 3))
        
        # First 3D Convolution block
        x = Conv3D(64, kernel_size=(3, 3, 3), padding='same', activation='relu')(inputs)
        x = BatchNormalization()(x)
        x = MaxPool3D(pool_size=(1, 2, 2))(x)
        
        # Second 3D Convolution block
        x = Conv3D(128, kernel_size=(3, 3, 3), padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
        x = MaxPool3D(pool_size=(2, 2, 2))(x)
        
        # Third 3D Convolution block
        x = Conv3D(256, kernel_size=(3, 3, 3), padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
        x = MaxPool3D(pool_size=(2, 2, 2))(x)
        
        # Fourth 3D Convolution block
        x = Conv3D(512, kernel_size=(3, 3, 3), padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
        x = MaxPool3D(pool_size=(2, 2, 2))(x)
        
        # Global pooling and dense layers
        x = GlobalAveragePooling3D()(x)
        x = Dense(512, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(0.5)(x)
        
        x = Dense(256, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(0.5)(x)
        
        outputs = Dense(num_classes, activation='softmax')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        
        # Compile with AdamW optimizer
        optimizer = AdamW(
            learning_rate=1e-4,
            weight_decay=1e-5
        )
        
        model.compile(
            optimizer=optimizer,
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        return model
    
    def train(self, video_dir, metadata_path, max_frames, epochs=50, batch_size=8):
        """Train the model"""
        try:
            # Validate directories and files
            if not os.path.exists(video_dir):
                raise ValueError(f"Video directory not found: {video_dir}")
            if not os.path.exists(metadata_path):
                raise ValueError(f"Metadata file not found: {metadata_path}")
                
            # Validate metadata structure
            if not self.validate_metadata(metadata_path):
                raise ValueError("Metadata validation failed")
            
            # Create output directories
            os.makedirs('models', exist_ok=True)
            os.makedirs('results', exist_ok=True)
            
            # Prepare data
            print("Preparing dataset...")
            X, y = self.prepare_data(video_dir, metadata_path, max_frames)
            
            if X is None or y is None:
                raise ValueError("Data preparation failed")
        
            
        except Exception as e:
            print(f"Error in training process: {str(e)}")
            raise
        
        # Split data
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.35, random_state=42, stratify=y
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
        )
        
        # Build model
        print("Building model...")
        self.model = self.build_model(num_classes=len(np.unique(y)))
        
        # Print model summary
        self.model.summary()
        
        # Callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_accuracy',
                patience=10,
                restore_best_weights=True
            ),
            ModelCheckpoint(
                filepath=f'models/3dcnn_model_{datetime.now().strftime("%Y%m%d_%H%M%S")}.keras',
                monitor='val_accuracy',
                save_best_only=True
            )
        ]
        
        # Train model
        print("Starting training...")
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks
        )
        
        # Evaluate and visualize
        self.evaluate(X_test, y_test, history)
        
        return history
    
    def evaluate(self, X_test, y_test, history):
        """Evaluate model and create visualizations"""
        # Test predictions
        y_pred = np.argmax(self.model.predict(X_test), axis=1)
        
        # Get original label names for visualization
        label_names = self.label_mapping['int_to_label']
        
        # Plot training history
        plt.figure(figsize=(15, 5))
        
        # Accuracy plot
        plt.subplot(1, 3, 1)
        plt.plot(history.history['accuracy'], label='Train')
        plt.plot(history.history['val_accuracy'], label='Validation')
        plt.title('Model Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        # Loss plot
        plt.subplot(1, 3, 2)
        plt.plot(history.history['loss'], label='Train')
        plt.plot(history.history['val_loss'], label='Validation')
        plt.title('Model Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
        # Confusion matrix with original labels
        plt.subplot(1, 3, 3)
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(
            cm, 
            annot=True, 
            fmt='d', 
            cmap='Blues',
            xticklabels=label_names,
            yticklabels=label_names
        )
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        
        plt.tight_layout()
        plt.savefig(f'results/3dcnn_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png')
        plt.close()
        
        # Save classification report with original labels
        report = classification_report(
            y_test, 
            y_pred,
            target_names=label_names
        )
        
        with open(f'results/3dcnn_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt', 'w') as f:
            f.write("Label Mapping:\n")
            for label, idx in self.label_mapping['label_to_int'].items():
                f.write(f"{label} -> Class {idx}\n")
            f.write("\nClassification Report:\n")
            f.write(report)
        
        # Print evaluation metrics
        test_loss, test_accuracy = self.model.evaluate(X_test, y_test)
        print(f"\nTest Loss: {test_loss:.4f}")
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print("\nLabel Mapping:")
        for label, idx in self.label_mapping['label_to_int'].items():
            print(f"{label} -> Class {idx}")
        print("\nClassification Report:")
        print(report)

: 

In [5]:
if __name__ == "__main__": 
    
    try:
        config = {
            'video_dir': '/Users/jeremiah/phd_thesis/data/main_data/videos',
            'metadata_path': '/Users/jeremiah/phd_thesis/data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx',
            'max_frames': 20,
            'frame_size': (224, 224),
            'epochs': 10,
            'batch_size': 8
        }
        
        # Initialize classifier
        classifier = Video3DClassifier(
            max_frames=config['max_frames'],
            frame_size=config['frame_size']
        )
        
        # Train model
        history = classifier.train(
            video_dir=config['video_dir'],
            metadata_path=config['metadata_path'],
            max_frames=config['max_frames'],
            epochs=config['epochs'],
            batch_size=config['batch_size']
        )
        
    except Exception as e:
        print(f"Error in main execution: {str(e)}")


Metadata Structure:
Unnamed: 0           int64
VideoID             object
Consent             object
Sharing             object
ParentID            object
ChildID             object
Age                  int64
Gender              object
PregnancyLength     object
Languages           object
JokeOrder           object
ParentFeedback     float64
UseOfMedia          object
Withdrawal            bool
DatabraryShare      object
JokeType            object
JokeNum              int64
RepNum               int64
HowFunny            object
Laugh               object
take                 int64
shortname          float64
dtype: object

Sample data:
   Unnamed: 0                                   VideoID   Consent     Sharing  \
0         993  NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4  accepted  scientific   
1         994  NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4  accepted  scientific   
2         995  NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4  accepted  scientific   
3         996  NFCK3A.joke1.rep3.

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d (Conv3D)                 │ (None, 20, 224, 224,   │         5,248 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 20, 224, 224,   │           256 │
│ (BatchNormalization)            │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 20, 112, 112,   │             0 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 20, 112, 112,   │       221,312 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 112, 112,   │           512 │
│ (BatchNormalization)            │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 10, 56, 56,     │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 10, 56, 56,     │       884,992 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 56, 56,     │         1,024 │
│ (BatchNormalization)            │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_2 (MaxPooling3D)  │ (None, 5, 28, 28, 256) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_3 (Conv3D)               │ (None, 5, 28, 28, 512) │     3,539,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 5, 28, 28, 512) │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_3 (MaxPooling3D)  │ (None, 2, 14, 14, 512) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling3d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling3D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,02

 Total params: 5,052,932 (19.28 MB)

 Trainable params: 5,049,476 (19.26 MB)

 Non-trainable params: 3,456 (13.50 KB)

Starting training...
Epoch 1/10


: 

: 